# 모기 비행 궤적 예측 AI 경진대회 베이스라인
- 4개 모델(1D-CNN, TCN, Transformer, LightGBM)과 5-Fold Stacking Ensemble 적용
- 물리 역학 기반 피처링, 3D Rotation 증강 기법 포함


## 1. 패키지 로드 및 설정 (Imports and Configuration)

In [17]:
# 1. 패키지 로드 및 설정 (Imports and Configuration)
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import Ridge
import math
import datetime
import warnings
warnings.filterwarnings('ignore')

CFG = {
    'DATA_DIR': './data',
    'SUBMIT_DIR': './submit',
    'EPOCHS': 20, 
    'BATCH_SIZE': 64,
    'LR': 1e-3,
    'SEED': 42,
    'N_FOLDS': 5,
    'TARGET_COLS': ['x', 'y', 'z'],
    'ENSEMBLE_SEEDS': [42, 52, 62, 72, 82],
}

os.makedirs(CFG['SUBMIT_DIR'], exist_ok=True)

## 1.1 seed_everything 함수

In [18]:
def seed_everything(seed):
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(CFG['SEED'])

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


Using device: mps


## 2. 데이터 로드 (Data Load)

In [19]:
# 2. 데이터 로드 (Data Load)
train_labels = pd.read_csv(f"{CFG['DATA_DIR']}/train_labels.csv")
train_ids = train_labels['id'].values
y_train = train_labels[['x', 'y', 'z']].values

y_train = train_labels[['x', 'y', 'z']].values


## 2.1 load_seqs 함수

In [20]:
def load_seqs(ids, folder):
    seqs = []
    for uid in ids:
        df = pd.read_csv(f"{CFG['DATA_DIR']}/{folder}/{uid}.csv")
        seqs.append(df[['x', 'y', 'z']].values)
    return np.array(seqs)

print("Loading train sequences...")
X_train = load_seqs(train_ids, 'train')

test_files = sorted(glob.glob(f"{CFG['DATA_DIR']}/test/*.csv"))
test_ids = [os.path.basename(f).split('.')[0] for f in test_files]

print("Loading test sequences...")
X_test = load_seqs(test_ids, 'test')
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Loading train sequences...
Loading test sequences...
Train shape: (10000, 11, 3)
Test shape: (10000, 11, 3)


## 3. 물리 역학 기반 피처 엔지니어링 (Physics-based Feature Engineering)

## 3.1 compute_features 함수

In [21]:
# 3. 물리 역학 기반 피처 엔지니어링 (Physics-based Feature Engineering)
# 속도, 가속도, 저크 및 3차원 구면 좌표계 변환을 통한 윈도우 통계량 추출
def compute_features(seq):
    # seq: (N, T, 3)
    eps = 1e-8

    # 1차, 2차, 3차 차분 (Velocity, Acceleration, Jerk)
    v = np.zeros_like(seq)
    v[:, 1:, :] = seq[:, 1:, :] - seq[:, :-1, :]

    a = np.zeros_like(seq)
    a[:, 2:, :] = v[:, 2:, :] - v[:, 1:-1, :]

    j = np.zeros_like(seq)
    j[:, 3:, :] = a[:, 3:, :] - a[:, 2:-1, :]

    # 3차원 구면 좌표계 변환
    r = np.sqrt(np.sum(seq**2, axis=-1))
    yaw = np.arctan2(seq[:, :, 1], seq[:, :, 0])
    pitch = np.arctan2(seq[:, :, 2], np.sqrt(seq[:, :, 0]**2 + seq[:, :, 1]**2))

    # 각속도
    yaw_v = np.zeros_like(yaw)
    yaw_v[:, 1:] = yaw[:, 1:] - yaw[:, :-1]

    pitch_v = np.zeros_like(pitch)
    pitch_v[:, 1:] = pitch[:, 1:] - pitch[:, :-1]

    # 기존 윈도우 통계량
    seq_mean = np.mean(seq, axis=1)
    seq_std = np.std(seq, axis=1)
    seq_max = np.max(seq, axis=1)
    seq_min = np.min(seq, axis=1)

    # [추가 피처 1] 시작-끝 변위(3축) + 변위 크기
    displacement = seq[:, -1, :] - seq[:, 0, :]
    displacement_norm = np.linalg.norm(displacement, axis=1, keepdims=True)

    # [추가 피처 2] 총 이동거리(Arc length)
    step_vec = np.diff(seq, axis=1)
    step_dist = np.linalg.norm(step_vec, axis=2)
    path_length = np.sum(step_dist, axis=1, keepdims=True)

    # [추가 피처 3] 속도 크기 분포 분위수(q10, q50, q90)
    speed_mag = np.linalg.norm(v, axis=2)
    speed_q = np.quantile(speed_mag, [0.1, 0.5, 0.9], axis=1).T

    # [추가 피처 4] 방향 변화량(턴 각도) 통계
    unit_step = step_vec / (np.linalg.norm(step_vec, axis=2, keepdims=True) + eps)
    if unit_step.shape[1] >= 2:
        turn_cos = np.sum(unit_step[:, 1:, :] * unit_step[:, :-1, :], axis=2)
        turn_cos = np.clip(turn_cos, -1.0, 1.0)
        turn_angle = np.arccos(turn_cos)
        turn_stats = np.stack([
            np.mean(turn_angle, axis=1),
            np.std(turn_angle, axis=1),
            np.max(turn_angle, axis=1),
        ], axis=1)
    else:
        turn_stats = np.zeros((seq.shape[0], 3), dtype=seq.dtype)

    # [추가 피처 5] 시간축 선형 추세 기울기(축별)
    t = np.arange(seq.shape[1], dtype=np.float32)
    t_centered = t - t.mean()
    denom = np.sum(t_centered**2) + eps
    trend_slope = np.sum(seq * t_centered[None, :, None], axis=1) / denom

    # 피처 병합
    seq_features = np.concatenate([
        seq, v, a, j,
        r[..., np.newaxis],
        yaw[..., np.newaxis],
        pitch[..., np.newaxis],
        yaw_v[..., np.newaxis],
        pitch_v[..., np.newaxis],
    ], axis=2)  # 총 17 features

    global_features = np.concatenate([
        seq_mean, seq_std, seq_max, seq_min,
        displacement, displacement_norm, path_length,
        speed_q, turn_stats, trend_slope,
    ], axis=1)  # 총 26 features

    return seq_features, global_features

In [22]:
print("Computing features...")
X_train_seq_f, X_train_global_f = compute_features(X_train)
X_test_seq_f, X_test_global_f = compute_features(X_test)

print("Sequence feature shape:", X_train_seq_f.shape)
print("Global feature shape:", X_train_global_f.shape)

# GBDT용 피처 구성 (시계열 평탄화 + 전역 피처 결합)
X_train_gbdt = np.concatenate([X_train_seq_f.reshape(X_train.shape[0], -1), X_train_global_f], axis=1)
X_test_gbdt = np.concatenate([X_test_seq_f.reshape(X_test.shape[0], -1), X_test_global_f], axis=1)
print("GBDT train feature shape:", X_train_gbdt.shape)
print("GBDT test feature shape:", X_test_gbdt.shape)

Computing features...
Sequence feature shape: (10000, 11, 17)
Global feature shape: (10000, 26)
GBDT train feature shape: (10000, 213)
GBDT test feature shape: (10000, 213)


## 4. 파이토치 Dataset 및 데이터 증강 (Dataset & Data Augmentation)

## 4.1 MosquitoDataset 클래스

In [7]:
# 4. 파이토치 Dataset 및 데이터 증강 (Dataset & Data Augmentation)
# Z축 기준 회전(3D Rotation) 및 가우시안 노이즈 주입
class MosquitoDataset(Dataset):
    def __init__(self, raw_seq, targets=None, is_train=False):
        self.raw_seq = torch.tensor(raw_seq, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32) if targets is not None else None
        self.is_train = is_train
        
    def __len__(self):
        return len(self.raw_seq)
        
    def __getitem__(self, idx):
        seq = self.raw_seq[idx].clone()
        y = self.targets[idx].clone() if self.targets is not None else None
        
        # 증강 기법 적용
        if self.is_train:
            # Random Z-axis Rotation
            angle = torch.rand(1).item() * 2 * np.pi
            cos_a = np.cos(angle)
            sin_a = np.sin(angle)
            rot_matrix = torch.tensor([
                [cos_a, -sin_a, 0],
                [sin_a,  cos_a, 0],
                [0,      0,     1]
            ], dtype=torch.float32)
            seq = torch.matmul(seq, rot_matrix.T)
            if y is not None:
                y = torch.matmul(y, rot_matrix.T)
            
            # Gaussian Noise 주입
            noise = torch.randn_like(seq) * 0.005 
            seq += noise
            
        # 개별 샘플에 대한 피처 계산
        seq_np = seq.unsqueeze(0).numpy()
        seq_f, global_f = compute_features(seq_np)
        
        seq_f = torch.tensor(seq_f.squeeze(0), dtype=torch.float32)
        global_f = torch.tensor(global_f.squeeze(0), dtype=torch.float32)
        
        if y is not None:
            return seq_f, global_f, y
        return seq_f, global_f


## 5. 모델 아키텍처 정의 (Model Architectures)

## 5.1 CNN1D 모델

In [8]:
# 5. 모델 아키텍처 정의 (Model Architectures)
# 1D-CNN, TCN, Transformer Encoder

# 5.1. 1D-CNN Model
class CNN1D(nn.Module):
    def __init__(self, num_seq_features=17, num_global_features=12):
        super().__init__()
        self.conv1 = nn.Conv1d(num_seq_features, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(64 + num_global_features, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )
    def forward(self, seq_x, global_x):
        x = seq_x.transpose(1, 2) # (B, Channels, SeqLen)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)
        out = self.fc(torch.cat([x, global_x], dim=1))
        return out

# 5.2. TCN Model


## 5.2 TCN 모델

In [9]:
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, num_seq_features=17, num_global_features=12):
        super().__init__()
        num_channels = [32, 64]
        layers = []
        for i in range(len(num_channels)):
            dilation_size = 2 ** i
            in_channels = num_seq_features if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size=3, stride=1, dilation=dilation_size, padding=(3-1) * dilation_size, dropout=0.1)]
        self.network = nn.Sequential(*layers)
        self.fc = nn.Sequential(
            nn.Linear(num_channels[-1] + num_global_features, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )
        
    def forward(self, seq_x, global_x):
        x = seq_x.transpose(1, 2)
        x = self.network(x)
        x = x[:, :, -1]
        out = self.fc(torch.cat([x, global_x], dim=1))
        return out

# 5.3. Transformer Encoder Model


## 5.3 Transformer Encoder 모델

In [10]:
class TransformerModel(nn.Module):
    def __init__(self, num_seq_features=17, num_global_features=12):
        super().__init__()
        self.d_model = 32
        self.embedding = nn.Linear(num_seq_features, self.d_model)
        encoder_layers = nn.TransformerEncoderLayer(d_model=self.d_model, nhead=4, dim_feedforward=64, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=2)
        self.fc = nn.Sequential(
            nn.Linear(self.d_model + num_global_features, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )
        
    def forward(self, seq_x, global_x):
        x = self.embedding(seq_x)
        
        # Positional Encoding (device-safe)
        pe = torch.zeros(1, x.size(1), self.d_model, device=x.device)
        position = torch.arange(x.size(1), device=x.device).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, self.d_model, 2, device=x.device).float() * (-math.log(10000.0) / self.d_model))
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        x = x + pe
        
        out = self.transformer_encoder(x)
        out = out.mean(dim=1) # Global Average Pooling
        out = self.fc(torch.cat([out, global_x], dim=1))
        return out


## 6. 학습 및 추론 함수 (Training and Inference Helpers)

## 6.1 train_model 함수

In [11]:
# 6. 학습 및 추론 함수 (Training and Inference Helpers)
def train_model(model_class, epochs, train_loader, val_loader, device):
    model = model_class().to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=CFG['LR'])
    
    best_loss = float('inf')
    best_model_wts = None
    
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            seq_x, glob_x, y = batch[0].to(device), batch[1].to(device), batch[2].to(device)
            optimizer.zero_grad()
            out = model(seq_x, glob_x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                seq_x, glob_x, y = batch[0].to(device), batch[1].to(device), batch[2].to(device)
                out = model(seq_x, glob_x)
                val_loss += criterion(out, y).item() * seq_x.size(0)
        val_loss /= len(val_loader.dataset)
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_wts = model.state_dict()
            
    model.load_state_dict(best_model_wts)
    return model

    return model


## 6.2 predict 함수

In [12]:
def predict(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            seq_x, glob_x = batch[0].to(device), batch[1].to(device)
            out = model(seq_x, glob_x)
            preds.append(out.cpu().numpy())
    return np.vstack(preds)


## 7. 안전모드 (Safe Mode)

In [13]:
# 안전모드: 강제 CPU 사용 및 학습 부담 축소
print('[ACTION] Applying safe-mode: force CPU and reduce training load')
import torch
device = torch.device('cpu')
print(f'Switched device -> {device}')
CFG['EPOCHS'] = 1
CFG['BATCH_SIZE'] = 8
print(f"CFG updated: EPOCHS={CFG['EPOCHS']}, BATCH_SIZE={CFG['BATCH_SIZE']}")

# CUDA 캐시 비우기 (있다면)
if torch.cuda.is_available():
    try:
        torch.cuda.empty_cache()
        print('[INFO] cuda.empty_cache() called')
    except Exception as e:
        print('[WARN] cuda.empty_cache() failed:', e)


[ACTION] Applying safe-mode: force CPU and reduce training load
Switched device -> cpu
CFG updated: EPOCHS=1, BATCH_SIZE=8


## 8. 5-Fold Cross Validation 및 앙상블을 위한 OOF 추출

In [23]:
# 7. LightGBM 타깃별 분리 학습 + 시드 앙상블 5-Fold CV
print("[INFO] Running target-wise LightGBM with seed ensemble")

n_splits = CFG['N_FOLDS']
n_samples = len(X_train_gbdt)
target_cols = CFG['TARGET_COLS']
ensemble_seeds = CFG['ENSEMBLE_SEEDS']

if n_samples < 2:
    raise ValueError(f"Not enough samples for CV: n_samples={n_samples}")

if n_samples < n_splits:
    print(f"[WARN] n_samples ({n_samples}) < n_splits ({n_splits}). Adjusting n_splits to {n_samples}")
    n_splits = n_samples

kf = KFold(n_splits=n_splits, shuffle=True, random_state=CFG['SEED'])

oof_gbdt = np.zeros_like(y_train, dtype=float)
test_gbdt = np.zeros((len(test_ids), len(target_cols)), dtype=float)

for fold, (trn_idx, val_idx) in enumerate(kf.split(X_train_gbdt), start=1):
    print(f"========== Fold {fold}/{n_splits} ==========")

    X_tr_g, y_tr_g = X_train_gbdt[trn_idx], y_train[trn_idx]
    X_va_g, y_va_g = X_train_gbdt[val_idx], y_train[val_idx]

    for t_idx, t_name in enumerate(target_cols):
        val_seed_preds = []
        test_seed_preds = []

        for seed in ensemble_seeds:
            # 멀티프로세싱 오류 방지를 위해 단일 스레드 고정
            model = lgb.LGBMRegressor(
                objective='regression',
                n_estimators=2000,
                learning_rate=0.015,
                num_leaves=63,
                min_child_samples=15,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_alpha=0.1,
                reg_lambda=0.5,
                random_state=seed + fold * 100 + t_idx,
                n_jobs=1,
                verbose=-1,
            )

            model.fit(
                X_tr_g, y_tr_g[:, t_idx],
                eval_set=[(X_va_g, y_va_g[:, t_idx])],
                eval_metric='l2',
                callbacks=[lgb.early_stopping(stopping_rounds=120, verbose=False)],
            )

            best_iter = model.best_iteration_ if model.best_iteration_ is not None else model.n_estimators
            val_seed_preds.append(model.predict(X_va_g, num_iteration=best_iter))
            test_seed_preds.append(model.predict(X_test_gbdt, num_iteration=best_iter))

        val_pred = np.mean(val_seed_preds, axis=0)
        test_pred = np.mean(test_seed_preds, axis=0)

        oof_gbdt[val_idx, t_idx] = val_pred
        test_gbdt[:, t_idx] += test_pred / n_splits

        fold_target_rmse = np.sqrt(mean_squared_error(y_va_g[:, t_idx], val_pred))
        print(f"  - Target {t_name} Fold RMSE: {fold_target_rmse:.4f}")

cv_rmse = np.sqrt(mean_squared_error(y_train, oof_gbdt))
oof_target_rmse = {
    t_name: np.sqrt(mean_squared_error(y_train[:, t_idx], oof_gbdt[:, t_idx]))
    for t_idx, t_name in enumerate(target_cols)
}

print(f"[DONE] Ensemble OOF RMSE: {cv_rmse:.4f}")
print("[DONE] Target-wise OOF RMSE:", oof_target_rmse)

# 제출 셀과의 호환을 위해 최종 예측 변수 유지
final_preds = test_gbdt.copy()

[INFO] Running target-wise LightGBM with seed ensemble
========== Fold 1/5 ==========
  - Target x Fold RMSE: 0.0204
  - Target y Fold RMSE: 0.0157
  - Target z Fold RMSE: 0.0130
========== Fold 2/5 ==========
  - Target x Fold RMSE: 0.0199
  - Target y Fold RMSE: 0.0169
  - Target z Fold RMSE: 0.0132
========== Fold 3/5 ==========
  - Target x Fold RMSE: 0.0191
  - Target y Fold RMSE: 0.0170
  - Target z Fold RMSE: 0.0146
========== Fold 4/5 ==========
  - Target x Fold RMSE: 0.0188
  - Target y Fold RMSE: 0.0176
  - Target z Fold RMSE: 0.0141
========== Fold 5/5 ==========
  - Target x Fold RMSE: 0.0188
  - Target y Fold RMSE: 0.0177
  - Target z Fold RMSE: 0.0165
[DONE] Ensemble OOF RMSE: 0.0170
[DONE] Target-wise OOF RMSE: {'x': np.float64(0.01940829697465223), 'y': np.float64(0.01701499071081568), 'z': np.float64(0.014331630745206915)}


## 9. LightGBM 단일 모델 결과 확인

In [24]:
# 8. LightGBM 단일(타깃별 + 시드 앙상블) 성능 확인
print("========== LightGBM Target-wise Seed Ensemble ==========")
oof_rmse = np.sqrt(mean_squared_error(y_train, oof_gbdt))
print(f"Overall OOF RMSE: {oof_rmse:.4f}")

if 'oof_target_rmse' in globals():
    for t_name, score in oof_target_rmse.items():
        print(f"{t_name} OOF RMSE: {score:.4f}")

# 최종 제출 예측값
final_preds = test_gbdt

========== LightGBM Target-wise Seed Ensemble ==========
Overall OOF RMSE: 0.0170
x OOF RMSE: 0.0194
y OOF RMSE: 0.0170
z OOF RMSE: 0.0143


## 10. 최종 제출 파일 생성 (Submission)

In [25]:
# 9. 최종 제출 파일 생성 (Submission)
submit = pd.DataFrame({
    'id': test_ids,
    'x': final_preds[:, 0],
    'y': final_preds[:, 1],
    'z': final_preds[:, 2]
})

now_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
submit_path = f"{CFG['SUBMIT_DIR']}/{now_str}.csv"
submit.to_csv(submit_path, index=False)
print(f"Submission saved successfully to: {submit_path}")


Submission saved successfully to: ./submit/20260527_124315.csv
